# 01 — Data Cleaning & Data Engineering

This notebook profiles the IBM Telco Customer Churn dataset, performs safe cleaning, validates the result, and saves a processed CSV.

**Target:** `Churn`


In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data_preprocessing import load_raw_data, clean_telco_data, save_cleaned_data


In [2]:
raw_path = ROOT / 'data' / 'raw' / 'Telco-Customer-Churn.csv'
df = load_raw_data(raw_path)
print('Shape:', df.shape)
display(df.head())


Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 1. Raw data profile


In [3]:
print('Columns:')
print(df.columns.tolist())
print('\nData types:')
display(df.dtypes.to_frame('dtype'))
print('\nMissing values:')
missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame('missing_count'))
print('Duplicate rows:', df.duplicated().sum())
print('Blank TotalCharges:', (df['TotalCharges'].astype(str).str.strip() == '').sum())
print('Unique Churn values:', df['Churn'].unique())


Columns:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

Data types:


,dtype
customerID,str
gender,str
SeniorCitizen,int64
Partner,str
Dependents,str
tenure,int64
PhoneService,str
MultipleLines,str
InternetService,str
OnlineSecurity,str



Missing values:


,missing_count


Duplicate rows: 0
Blank TotalCharges: 11
Unique Churn values: <ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str


## 2. Cleaning

`TotalCharges` is converted to numeric. Blank values become missing and are then filled with `0.0`, which is appropriate for customers whose accumulated charges are blank at the start of service.


In [4]:
cleaned = clean_telco_data(df)
print('Cleaned shape:', cleaned.shape)
print('Duplicate rows after cleaning:', cleaned.duplicated().sum())
display(cleaned.head())


Cleaned shape: (7043, 21)
Duplicate rows after cleaning: 0


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 3. Validation


In [5]:
assert cleaned['customerID'].notna().all()
assert cleaned['customerID'].is_unique
assert cleaned['TotalCharges'].dtype.kind in 'fi'
assert set(cleaned['Churn'].dropna().unique()) <= {'Yes', 'No'}
print('All Stage 2 validation checks passed.')


All Stage 2 validation checks passed.


In [6]:
processed_path = ROOT / 'data' / 'processed' / 'telco_customer_churn_cleaned.csv'
save_cleaned_data(cleaned, processed_path)
print(f'Saved: {processed_path}')


Saved: C:\Users\sakas\Downloads\customer-churn-data-science-stage1\customer-churn-data-science\data\processed\telco_customer_churn_cleaned.csv
